# Gemma 4 LiteRT Comparison and Eval

This notebook evaluates LiteRT-LM artifacts against the merged Hugging Face checkpoint produced by the VLM merge notebook.

## What this notebook does
- Uses the merged safetensors checkpoint as the canonical reference artifact
- Runs LiteRT-LM smoke tests for text generation
- Runs LiteRT-LM benchmarks
- Runs agentic tool-calling tests with a local preset
- Provides an optional multimodal LiteRT Python API test if `litert_lm` is available in the active kernel

## What this notebook does not do
- It does not merge LoRA into Gemma 4; that already happens in the A100 VLM notebook
- It does not claim a verified in-notebook HF checkpoint -> `.litertlm` conversion path for your custom legal model

## Practical workflow
1. Produce and validate `gemma4-legal-vlm-merged/` in the VLM notebook
2. Point this notebook at either:
   - a published LiteRT model for baseline runtime testing, or
   - a custom `.litertlm` artifact once you have packaged one
3. Compare text, benchmark, tool-calling, and optional multimodal behavior

In [1]:
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
# This notebook lives under scripts/unsloth-training, so walk back to the repo root.
ROOT = NOTEBOOK_DIR.parent.parent
MERGED_HF_DIR = ROOT / 'gemma4-legal-vlm-merged'
RESULTS_DIR = NOTEBOOK_DIR / 'litert-eval-results'
RESULTS_DIR.mkdir(exist_ok=True)

IS_WINDOWS = platform.system().lower().startswith('win')
HAS_WSL = IS_WINDOWS and shutil.which('wsl.exe') is not None
LOCAL_LITERT = shutil.which('litert-lm')

if HAS_WSL:
    LITERT_RUNNER = ['wsl.exe', 'bash', '-lc']
    LITERT_BINARY = '~/.local/bin/litert-lm'
elif LOCAL_LITERT:
    LITERT_RUNNER = None
    LITERT_BINARY = LOCAL_LITERT
else:
    LITERT_RUNNER = None
    LITERT_BINARY = 'litert-lm'

def run_litert_shell(command, check=True):
    if LITERT_RUNNER is not None:
        full_cmd = LITERT_RUNNER + [command]
    else:
        full_cmd = shlex.split(command)

    print('Running:', command)
    result = subprocess.run(full_cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result

print('Notebook dir:', NOTEBOOK_DIR)
print('Repo root:', ROOT)
print('Merged HF dir exists:', MERGED_HF_DIR.exists())
print('Windows:', IS_WINDOWS)
print('WSL available:', HAS_WSL)
print('Local litert-lm binary:', LOCAL_LITERT or 'not on PATH')

Notebook dir: c:\Users\james\Videos\deeds-web-app\scripts\unsloth-training
Repo root: c:\Users\james\Videos\deeds-web-app
Merged HF dir exists: False
Windows: True
WSL available: True
Local litert-lm binary: not on PATH


## 1. Choose the LiteRT Artifact

Use the published Gemma 4 LiteRT model for runtime sanity first, then swap in a custom `.litertlm` artifact when you have one.

- `HF_BASELINE_*` is the known-good baseline path
- `CUSTOM_LITERT_PATH` can point to a local `.litertlm` file
- `CUSTOM_HF_REPO` and `CUSTOM_HF_FILE` can point to a custom uploaded LiteRT artifact

In [2]:
HF_BASELINE_REPO = 'litert-community/gemma-4-E2B-it-litert-lm'
HF_BASELINE_FILE = 'gemma-4-E2B-it.litertlm'
CUSTOM_LITERT_PATH = ''
CUSTOM_HF_REPO = ''
CUSTOM_HF_FILE = ''
BACKEND = 'gpu'

def resolve_model_reference():
    if CUSTOM_LITERT_PATH:
        return {
            'mode': 'local-file',
            'reference': CUSTOM_LITERT_PATH,
            'repo': None,
            'label': Path(CUSTOM_LITERT_PATH).name
        }
    if CUSTOM_HF_REPO and CUSTOM_HF_FILE:
        return {
            'mode': 'hf-custom',
            'reference': CUSTOM_HF_FILE,
            'repo': CUSTOM_HF_REPO,
            'label': f'{CUSTOM_HF_REPO}:{CUSTOM_HF_FILE}'
        }
    return {
        'mode': 'hf-baseline',
        'reference': HF_BASELINE_FILE,
        'repo': HF_BASELINE_REPO,
        'label': f'{HF_BASELINE_REPO}:{HF_BASELINE_FILE}'
    }

MODEL_REF = resolve_model_reference()
print(json.dumps(MODEL_REF, indent=2))

{
  "mode": "hf-baseline",
  "reference": "gemma-4-E2B-it.litertlm",
  "repo": "litert-community/gemma-4-E2B-it-litert-lm",
  "label": "litert-community/gemma-4-E2B-it-litert-lm:gemma-4-E2B-it.litertlm"
}


## 2. Text Smoke Test

This validates that LiteRT-LM can run the selected model artifact and answer a legal reasoning prompt.

In [ ]:
TEXT_PROMPT = (
    'Explain whether a notarized affidavit is self-authenticating, how hearsay objections may still apply, '
    'and what additional foundation a prosecutor may need at trial.'
)

if MODEL_REF['repo']:
    cmd = (
        f"{LITERT_BINARY} run --backend={BACKEND} --from-huggingface-repo={MODEL_REF['repo']} "
        f"{MODEL_REF['reference']} --prompt={shlex.quote(TEXT_PROMPT)}"
    )
else:
    cmd = f"{LITERT_BINARY} run --backend={BACKEND} {shlex.quote(MODEL_REF['reference'])} --prompt={shlex.quote(TEXT_PROMPT)}"

result = run_litert_shell(cmd)
(RESULTS_DIR / 'text_smoke.txt').write_text(result.stdout, encoding='utf-8')
print('Saved:', RESULTS_DIR / 'text_smoke.txt')

Running: ~/.local/bin/litert-lm run --backend=gpu --from-huggingface-repo=litert-community/gemma-4-E2B-it-litert-lm gemma-4-E2B-it.litertlm --prompt='Explain whether a notarized affidavit is self-authenticating, how hearsay objections may still apply, and what additional foundation a prosecutor may need at trial.'


## 3. Benchmark

Run a quick benchmark so you can compare LiteRT throughput against TRT-LLM, TurboQuant, or HF-based local generation later.

In [ ]:
if MODEL_REF['repo']:
    cmd = (
        f"{LITERT_BINARY} benchmark --backend={BACKEND} --from-huggingface-repo={MODEL_REF['repo']} "
        f"{MODEL_REF['reference']} --prefill_tokens=256 --decode_tokens=128"
    )
else:
    cmd = (
        f"{LITERT_BINARY} benchmark --backend={BACKEND} "
        f"{shlex.quote(MODEL_REF['reference'])} --prefill_tokens=256 --decode_tokens=128"
    )

result = run_litert_shell(cmd)
(RESULTS_DIR / 'benchmark.txt').write_text(result.stdout, encoding='utf-8')
print('Saved:', RESULTS_DIR / 'benchmark.txt')

## 4. Tool Calling / Agentic Test

LiteRT-LM supports automatic tool use with presets. This test gives you a comparable agentic path to measure against TRT text serving and your merged HF branch.

In [ ]:
preset_code = '''
import datetime

def glossary_search(query: str) -> str:
    """Searches a tiny local legal glossary."""
    glossary = {
        'chain of custody': 'The documented control, transfer, analysis, and disposition of evidence.',
        'habeas corpus': 'A legal action challenging unlawful detention.',
        'stare decisis': 'The doctrine of following precedent.'
    }
    key = query.strip().lower()
    return glossary.get(key, f'No glossary match for: {query}')

def get_current_time() -> str:
    """Returns the current local time."""
    return datetime.datetime.now().isoformat(timespec='seconds')

system_instruction = 'You are a legal AI assistant with access to tools. Use tools when they materially improve accuracy.'
tools = [glossary_search, get_current_time]
'''

preset_path = RESULTS_DIR / 'litert_preset.py'
preset_path.write_text(preset_code, encoding='utf-8')
tool_prompt = 'Use the glossary_search tool to define chain of custody, then explain why it matters in a criminal evidence challenge.'

if MODEL_REF['repo']:
    cmd = (
        f"{LITERT_BINARY} run --backend={BACKEND} --from-huggingface-repo={MODEL_REF['repo']} "
        f"{MODEL_REF['reference']} --preset={shlex.quote(str(preset_path))} --prompt={shlex.quote(tool_prompt)}"
    )
else:
    cmd = (
        f"{LITERT_BINARY} run --backend={BACKEND} {shlex.quote(MODEL_REF['reference'])} "
        f"--preset={shlex.quote(str(preset_path))} --prompt={shlex.quote(tool_prompt)}"
    )

result = run_litert_shell(cmd)
(RESULTS_DIR / 'tool_calling.txt').write_text(result.stdout, encoding='utf-8')
print('Saved:', RESULTS_DIR / 'tool_calling.txt')

## 5. Optional Python API Multimodal Test

Use this only if the active Python kernel has the `litert_lm` package installed and your selected `.litertlm` model actually supports multimodality.

If this cell is skipped, keep VLM validation in the merged HF or GGUF branches and use LiteRT for text and tool-call comparisons only.

In [ ]:
try:
    import litert_lm
    from PIL import Image
except Exception as exc:
    print('Skipping multimodal Python API test:', exc)
    litert_lm = None

if litert_lm is not None:
    if MODEL_REF['mode'] != 'local-file':
        print('Multimodal Python API test expects a local `.litertlm` file. Set CUSTOM_LITERT_PATH first.')
    else:
        test_image_path = RESULTS_DIR / 'litert_test_image.jpg'
        Image.new('RGB', (512, 512), (220, 220, 220)).save(test_image_path)

        with (
            litert_lm.Engine(
                MODEL_REF['reference'],
                backend=litert_lm.Backend.GPU,
                vision_backend=litert_lm.Backend.CPU
            ) as engine,
            engine.create_conversation() as conversation,
        ):
            message = conversation.send_message({
                'role': 'user',
                'content': [
                    {'type': 'image', 'path': str(test_image_path)},
                    {'type': 'text', 'text': 'Describe this placeholder image and explain what legal document evidence is missing.'}
                ]
            })

        output_text = json.dumps(message, indent=2)
        print(output_text)
        (RESULTS_DIR / 'multimodal_python_api.json').write_text(output_text, encoding='utf-8')
        print('Saved:', RESULTS_DIR / 'multimodal_python_api.json')

## 6. Comparison Notes

Use the saved outputs in `litert-eval-results/` to compare:
- LiteRT text latency and output style
- LiteRT benchmark throughput
- LiteRT tool-calling behavior
- merged HF VLM output from the A100 notebook
- TRT-LLM text-serving output from the Triton branch

A practical comparison matrix is:
1. Text legal answer quality
2. Tool-calling correctness
3. Tokens per second / time to first token
4. VLM support status
5. Deployment complexity

In [ ]:
summary = {
    'merged_hf_dir_exists': MERGED_HF_DIR.exists(),
    'selected_model': MODEL_REF,
    'results_dir': str(RESULTS_DIR),
    'expected_outputs': [
        'text_smoke.txt',
        'benchmark.txt',
        'tool_calling.txt',
        'multimodal_python_api.json (optional)'
    ]
}
print(json.dumps(summary, indent=2))
(RESULTS_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')